In [9]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [10]:
!pip install -q open_clip_torch torch torchvision

In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"\nGPU Name: {torch.cuda.get_device_name(0)}")


Using device: cuda

GPU Available: True

GPU Name: Tesla P100-PCIE-16GB


In [12]:
import open_clip

model_name = "ViT-B-32"
print(f"\nLoading CLIP Model: {model_name}")

model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained='openai')
tokenizer = open_clip.get_tokenizer(model_name)
model = model.to(device)

print(f"\nModel Loaded Successfully")


Loading CLIP Model: ViT-B-32

Model Loaded Successfully


In [13]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image

class ImageTextDataset(Dataset):
    def __init__(self, csv_path, preprocess):
        df = pd.read_csv(csv_path)
        self.images = df['image_path'].tolist()
        self.texts = df['text'].astype(str).tolist()
        self.preprocess = preprocess

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        img = self.preprocess(img)
        txt = self.texts[idx]
        return img, txt

In [14]:
from torchvision.datasets import CocoCaptions
import os
import pandas as pd
from tqdm import tqdm
import kagglehub

path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print(f"\nPath to Dataset Files: {path}")

coco2017_path = os.path.join(path, "coco2017")
coco_root = os.path.join(coco2017_path, "train2017")
ann_file = os.path.join(coco2017_path, "annotations/captions_train2017.json")
train_csv_path = "dataset_train.csv"

print()
coco = CocoCaptions(root=coco_root, annFile=ann_file)
print()

if not os.path.exists(train_csv_path):
    image_paths = []
    texts = []

    for idx in tqdm(range(len(coco)), desc="Processing COCO Captions"):
        img, captions = coco[idx]
        img_path = os.path.join(coco_root, coco.coco.imgs[coco.ids[idx]]['file_name'])
        for caption in captions:
            image_paths.append(img_path)
            texts.append(caption)

    df_train = pd.DataFrame({"image_path": image_paths, "text": texts})
    df_train = df_train.sample(frac=1, random_state=501).reset_index(drop=True) 
    df_train.to_csv(train_csv_path, index=False)
    print(f"\nSaved CSV To: {train_csv_path}")
else:
    print(f"\nCSV File Already Exists: {train_csv_path}")
    df_train = pd.read_csv(train_csv_path)


Path to Dataset Files: /kaggle/input/coco-2017-dataset

loading annotations into memory...
Done (t=0.94s)
creating index...
index created!



Processing COCO Captions: 100%|██████████| 118287/118287 [14:26<00:00, 136.50it/s]



Saved CSV To: dataset_train.csv


In [15]:
print("Dataset Shape: ", df_train.shape)

Dataset Shape:  (591753, 2)


In [16]:
display(df_train.head(10))

                                          image_path  \
0  /kaggle/input/coco-2017-dataset/coco2017/train...   
1  /kaggle/input/coco-2017-dataset/coco2017/train...   
2  /kaggle/input/coco-2017-dataset/coco2017/train...   
3  /kaggle/input/coco-2017-dataset/coco2017/train...   
4  /kaggle/input/coco-2017-dataset/coco2017/train...   
5  /kaggle/input/coco-2017-dataset/coco2017/train...   
6  /kaggle/input/coco-2017-dataset/coco2017/train...   
7  /kaggle/input/coco-2017-dataset/coco2017/train...   
8  /kaggle/input/coco-2017-dataset/coco2017/train...   
9  /kaggle/input/coco-2017-dataset/coco2017/train...   

                                                text  
0  A bathroom with a sink, medicine cabinet and m...  
1                 People on skis are on a ski slope.  
2  there are three different lights that show if ...  
3   A group of men in bondage gear standing outside.  
4  A group of white vases sitting behind a glass ...  
5  a herd of elephants walking through a marsh land  

In [17]:
print(f"\nTotal Images in our Training Set: {len(os.listdir(coco_root))}")
print(f"\nTotal (Image, Text) Pairs in our CSV Train Dataset: {len(pd.read_csv(train_csv_path))}")


Total Images in our Training Set: 118287

Total (Image, Text) Pairs in our CSV Train Dataset: 591753


In [18]:
import time
import pandas as pd
import os

def mine_hard_negatives(dataset, loader, img_to_idx, unique_paths, unique_indices, batch_size=256, log_dir="logs"):
    H_paths = []
    os.makedirs(log_dir, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    log_path = os.path.join(log_dir, f"hard_negatives_logs_{timestamp}.csv")

    log_data = []

    with torch.no_grad():
        img_feats = []
        for b in tqdm(range(0, len(unique_paths), batch_size), desc="Encoding Unique Images"):
            batch_paths = unique_paths[b:b+batch_size]
            batch_imgs = []
            for p in batch_paths:
                batch_imgs.append(dataset[img_to_idx[p]][0].unsqueeze(0))
            batch_imgs = torch.cat(batch_imgs).to(device)
            feats = model.encode_image(batch_imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            img_feats.append(feats)
        img_feats = torch.cat(img_feats)
        print("\nUnique Images Features Shape:", img_feats.shape)

        all_txt_feats = []
        all_queries = []
        for _, texts in loader:
            tokens = tokenizer(texts).to(device)
            txt_feats = model.encode_text(tokens)
            txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
            all_txt_feats.append(txt_feats)
            all_queries.extend(texts)
        all_txt_feats = torch.cat(all_txt_feats)
        print("\nTexts Features Shape:", all_txt_feats.shape)

        logits = model.logit_scale.exp() * (all_txt_feats @ img_feats.t())

        for i in tqdm(range(logits.size(0)), desc="Processing Queries"):
            scores = logits[i]
            sorted_scores, sorted_idx = torch.sort(scores, descending=True)
            sorted_paths = [unique_paths[idx] for idx in sorted_idx]

            pos_path = dataset.images[i]
            pos_rank = sorted_paths.index(pos_path)
            pos_score = scores[unique_paths.index(pos_path)]
            query_text = all_queries[i]

            drops = sorted_scores[:-1] - sorted_scores[1:]
            top2 = torch.topk(drops, 2).indices
            drop_start, drop_end = min(top2).item(), max(top2).item()

            drop_band_paths = []
            for j, path in enumerate(sorted_paths[drop_start:drop_end+1]):
                if j + drop_start > pos_rank and scores[unique_paths.index(path)] < pos_score:
                    drop_band_paths.append(path)

            if drop_band_paths:
                hard_path = random.choice(drop_band_paths)
            else:
                hard_path = sorted_paths[-1]

            hard_rank = sorted_paths.index(hard_path)
            hard_score = scores[unique_paths.index(hard_path)]

            log_data.append({
                "caption_idx": i,
                "query_text": query_text,
                "pos_rank": pos_rank,
                "pos_path": pos_path,
                "pos_score": float(pos_score),
                "hard_neg_rank": hard_rank,
                "hard_neg_path": hard_path,
                "hard_neg_score": float(hard_score)
            })

            H_paths.append(hard_path)

    df_log = pd.DataFrame(log_data)
    df_log.to_csv(log_path, index=False)
    print("\nHard Negatives Image Features Shape:", len(H_paths))
    print(f"\nHard Negatives Log Saved To: {log_path}")
    return H_paths

In [19]:
import random
from tqdm import tqdm
import pandas as pd
from torch.utils.data import DataLoader

def train(train_csv_path, batch_size=64, epochs=3, lr=1e-5, num_workers=4,
          freeze_visual=False, freeze_text=True, updates=5):

    if freeze_visual:
        for p in model.visual.parameters(): p.requires_grad = False
    if freeze_text:
        for p in model.transformer.parameters(): p.requires_grad = False

    dataset = ImageTextDataset(train_csv_path, preprocess)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    out_prefix = "clip_ft"

    print(f"\nTotal Batches: {len(loader)} Batches * {batch_size} Units\n")

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=lr, weight_decay=1e-4)

    img_to_idx = {}
    for i, path in enumerate(dataset.images):
        if path not in img_to_idx:
            img_to_idx[path] = i
    
    unique_paths = list(img_to_idx.keys())
    unique_indices = list(img_to_idx.values())

    model.train()
    loss_history = {}

    for epoch in range(epochs):
        epoch_path = f"{out_prefix}_ep_{epoch+1}.pth"
        if os.path.exists(epoch_path):
            model.load_state_dict(torch.load(epoch_path, map_location=device))
            print(f"\nSkipping Epoch and Loading Model from Epoch: {epoch+1}")
            continue

        total_loss = 0.0
        if epoch == 0:
            H_paths = []
            shuffled = list(range(len(dataset)))
            while True:
                random.shuffle(shuffled)
                if all(h != i for i, h in enumerate(shuffled)):
                    break
            H_paths = [dataset.images[h] for h in shuffled]
        elif epoch % max(1, epochs // updates) == 0:
            print(f"\nEpoch {epoch+1}: Mining Hard Negatives")
            H_paths = mine_hard_negatives(dataset, loader, img_to_idx, unique_paths, unique_indices)

        H_indices = [img_to_idx[p] for p in H_paths]

        df = pd.DataFrame({
            "image_path": dataset.images,
            "text": dataset.texts,
            f"hard_neg_epoch_{epoch+1}": H_paths
        })
        df.to_csv(f"hard_negatives_ep_{epoch}.csv", index=False)

        for batch_idx, (imgs, texts) in enumerate(
            tqdm(loader, desc=f"Processing Batches Epoch {epoch+1}/{epochs}", leave=False)
        ):
            imgs, tokens = imgs.to(device), tokenizer(texts).to(device)
            optimizer.zero_grad()
                
            img_feats = model.encode_image(imgs)
            txt_feats = model.encode_text(tokens)
            
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
            
            batch_H = H_indices[batch_idx*batch_size : batch_idx*batch_size + len(imgs)]
            neg_imgs = torch.stack([dataset[idx][0] for idx in batch_H]).to(device)
            
            neg_img_feats = model.encode_image(neg_imgs)
            neg_img_feats = neg_img_feats / neg_img_feats.norm(dim=-1, keepdim=True)
            
            pos_scores = (img_feats * txt_feats).sum(dim=-1)
            neg_scores = (neg_img_feats * txt_feats).sum(dim=-1)
            loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        loss_history[epoch+1] = total_loss/len(loader)
        print(f"\nEpoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.4f}")
        torch.save(model.state_dict(), epoch_path)
        print(f"\nEpoch Model Saved To: {epoch_path}")

    model_path = f"{out_prefix}_final.pth"
    torch.save(model.state_dict(), model_path)
    print("\nTraining Completed")
    print(f"\nModel Saved To: {model_path}")

    return loss_history

In [20]:
import pandas as pd

csv_path = "dataset_train.csv"
k = 0.01

df = pd.read_csv(csv_path)
subset_len = int(len(df) * k)
df_subset = df.iloc[:subset_len]

train_csv_path = f"dataset_train_{int(k*100)}.csv"
df_subset.to_csv(train_csv_path, index=False)

In [21]:
print("Dataset Shape: ", df_subset.shape)

Dataset Shape:  (5917, 2)


In [22]:
display(df_subset.head(10))

                                          image_path  \
0  /kaggle/input/coco-2017-dataset/coco2017/train...   
1  /kaggle/input/coco-2017-dataset/coco2017/train...   
2  /kaggle/input/coco-2017-dataset/coco2017/train...   
3  /kaggle/input/coco-2017-dataset/coco2017/train...   
4  /kaggle/input/coco-2017-dataset/coco2017/train...   
5  /kaggle/input/coco-2017-dataset/coco2017/train...   
6  /kaggle/input/coco-2017-dataset/coco2017/train...   
7  /kaggle/input/coco-2017-dataset/coco2017/train...   
8  /kaggle/input/coco-2017-dataset/coco2017/train...   
9  /kaggle/input/coco-2017-dataset/coco2017/train...   

                                                text  
0  A bathroom with a sink, medicine cabinet and m...  
1                 People on skis are on a ski slope.  
2  there are three different lights that show if ...  
3   A group of men in bondage gear standing outside.  
4  A group of white vases sitting behind a glass ...  
5  a herd of elephants walking through a marsh land  

In [23]:
train_csv_path = f"dataset_train_{int(k*100)}.csv"
loss_history = train(train_csv_path, epochs = 5)


Total Batches: 93 Batches * 64 Units




Epoch 1/5 | Loss: 0.5421

Epoch Model Saved To: clip_ft_ep_1.pth

Epoch 2: Mining Hard Negatives


Encoding Unique Images: 100%|██████████| 23/23 [00:50<00:00,  2.20s/it]


Unique Images Features Shape: torch.Size([5814, 512])



Texts Features Shape: torch.Size([5917, 512])


Processing Queries: 100%|██████████| 5917/5917 [19:14<00:00,  5.13it/s]


... (16 outputs left out)


Texts Features Shape: torch.Size([5917, 512])


Processing Queries: 100%|██████████| 5917/5917 [22:17<00:00,  4.42it/s]



Hard Negatives Image Features Shape: 5917

Hard Negatives Log Saved To: logs/hard_negatives_logs_20251103_174929.csv



Epoch 5/5 | Loss: 0.4574

Epoch Model Saved To: clip_ft_ep_5.pth

Training Completed

Model Saved To: clip_ft_final.pth


In [24]:
import matplotlib.pyplot as plt

print(f"\nTraining Results on {k*100:.2f}% Train Dataset:")
print("\nLoss History: ", loss_history)

plt.figure(figsize=(6,4))
plt.plot(list(loss_history.keys()), list(loss_history.values()), marker='o')
plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("Loss History")
plt.grid(True)
plt.show()


Training Results on 1.00% Train Dataset:

Loss History:  {1: 0.5420566082641642, 2: 0.5666867686856177, 3: 0.5336048856217374, 4: 0.48079405100114886, 5: 0.45735172430674237}


<Figure size 600x400 with 1 Axes>

In [25]:
from torchvision.datasets import CocoCaptions
import os
import pandas as pd
from tqdm import tqdm
import kagglehub

path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print(f"\nPath to Dataset Files: {path}")

coco2017_path = os.path.join(path, "coco2017")
coco_root = os.path.join(coco2017_path, "val2017")
ann_file = os.path.join(coco2017_path, "annotations/captions_val2017.json")
test_csv_path = "dataset_test.csv"

print()
coco = CocoCaptions(root=coco_root, annFile=ann_file)
print()

if not os.path.exists(test_csv_path):
    image_paths = []
    texts = []

    for idx in tqdm(range(len(coco)), desc="Processing COCO Captions"):
        img, captions = coco[idx]
        img_path = os.path.join(coco_root, coco.coco.imgs[coco.ids[idx]]['file_name'])
        for caption in captions:
            image_paths.append(img_path)
            texts.append(caption)

    df_test = pd.DataFrame({"image_path": image_paths, "text": texts})
    df_test = df_test.sample(frac=1, random_state=501).reset_index(drop=True) 
    df_test.to_csv(test_csv_path, index=False)
    print(f"\nSaved CSV To: {test_csv_path}")
else:
    print(f"\nCSV File Already Exists: {test_csv_path}")
    df_test = pd.read_csv(test_csv_path)


Path to Dataset Files: /kaggle/input/coco-2017-dataset

loading annotations into memory...
Done (t=0.12s)
creating index...
index created!



Processing COCO Captions: 100%|██████████| 5000/5000 [01:01<00:00, 81.93it/s]



Saved CSV To: dataset_test.csv


In [26]:
print("Dataset Shape: ", df_test.shape)

Dataset Shape:  (25014, 2)


In [27]:
display(df_test.head(10))

                                          image_path  \
0  /kaggle/input/coco-2017-dataset/coco2017/val20...   
1  /kaggle/input/coco-2017-dataset/coco2017/val20...   
2  /kaggle/input/coco-2017-dataset/coco2017/val20...   
3  /kaggle/input/coco-2017-dataset/coco2017/val20...   
4  /kaggle/input/coco-2017-dataset/coco2017/val20...   
5  /kaggle/input/coco-2017-dataset/coco2017/val20...   
6  /kaggle/input/coco-2017-dataset/coco2017/val20...   
7  /kaggle/input/coco-2017-dataset/coco2017/val20...   
8  /kaggle/input/coco-2017-dataset/coco2017/val20...   
9  /kaggle/input/coco-2017-dataset/coco2017/val20...   

                                                text  
0  A busy intersection with cars waiting for a re...  
1  A couple of beds sitting next to each other in...  
2            a man at the beach trying to fly a kite  
3             A tree stands net to an old wood house  
4  A white plate topped with a pizza cut into 6 s...  
5       Graffiti on a building next to street signs. 

In [28]:
print(f"\nTotal Images in our Testing Set: {len(os.listdir(coco_root))}")
print(f"\nTotal (Image, Text) Pairs in our CSV Test Dataset: {len(pd.read_csv(test_csv_path))}")


Total Images in our Testing Set: 5000

Total (Image, Text) Pairs in our CSV Test Dataset: 25014


In [29]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import json
import numpy as np
from datetime import datetime
import open_clip

@torch.no_grad()
def test(test_csv_path, model_path, batch_size=64, top_k=5):
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=None)
    tokenizer = open_clip.get_tokenizer(model_name)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    dataset = ImageTextDataset(test_csv_path, preprocess)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    img_to_idx, unique_paths = {}, []
    for i, path in enumerate(dataset.images):
        if path not in img_to_idx:
            img_to_idx[path] = i
            unique_paths.append(path)

    unique_img_feats = []
    for b in tqdm(range(0, len(unique_paths), batch_size), desc="Encoding Unique Images"):
        batch_paths = unique_paths[b:b+batch_size]
        batch_imgs = torch.stack([dataset[img_to_idx[p]][0] for p in batch_paths]).to(device)
        feats = model.encode_image(batch_imgs)
        feats /= feats.norm(dim=-1, keepdim=True)
        unique_img_feats.append(feats)
    unique_img_feats = torch.cat(unique_img_feats)
    print("\nUnique Images Features Shape:", unique_img_feats.shape)

    txt_feats = []
    for _, texts in tqdm(loader, desc="Encoding Texts"):
        tokens = tokenizer(list(texts)).to(device)
        feats = model.encode_text(tokens)
        feats /= feats.norm(dim=-1, keepdim=True)
        txt_feats.append(feats)
    txt_feats = torch.cat(txt_feats)
    print("\nTexts Features Shape:", txt_feats.shape)

    logits = model.logit_scale.exp() * (txt_feats @ unique_img_feats.t())

    results, ranks = [], []
    for i, text in enumerate(tqdm(dataset.texts, desc="Processing Queries")):
        scores = logits[i]
        sorted_idx = scores.argsort(descending=True)
        sorted_paths = [unique_paths[j] for j in sorted_idx.tolist()]

        gt_path = dataset.images[i]
        gt_rank = sorted_paths.index(gt_path) + 1
        gt_score = float(scores[sorted_idx[gt_rank-1]])
        ranks.append(gt_rank)

        retrieved = [
            {
                "rank": r+1,
                "path": unique_paths[idx],
                "score": float(scores[idx])
            } for r, idx in enumerate(sorted_idx[:top_k])
        ]

        results.append({
            "query": text,
            "gt_path": gt_path,
            "gt_rank": gt_rank,
            "gt_score": gt_score,
            "retrieved": retrieved
        })

    ranks = np.array(ranks)
    metrics = {"MRR": float(np.mean(1 / ranks)), "Results": results}
    for k_val in range(1, top_k+1):
        metrics[f"Precision@{k_val}"] = float(np.mean(ranks <= k_val))    
        metrics[f"Recall@{k_val}"] = float(np.mean(ranks <= k_val)) 

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    metrics_path = f"clip_test_metrics_{timestamp}.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)

    print(f"\nMetrics Saved To: {metrics_path}")
    return metrics

In [30]:
import pandas as pd

csv_path = "dataset_test.csv"
k = 0.25

df = pd.read_csv(csv_path)
subset_len = int(len(df) * k)
df_subset = df.iloc[:subset_len]

train_csv_path = f"dataset_test_{int(k*100)}.csv"
df_subset.to_csv(train_csv_path, index=False)

In [31]:
print("Dataset Shape: ", df_subset.shape)

Dataset Shape:  (6253, 2)


In [32]:
display(df_subset.head(10))

                                          image_path  \
0  /kaggle/input/coco-2017-dataset/coco2017/val20...   
1  /kaggle/input/coco-2017-dataset/coco2017/val20...   
2  /kaggle/input/coco-2017-dataset/coco2017/val20...   
3  /kaggle/input/coco-2017-dataset/coco2017/val20...   
4  /kaggle/input/coco-2017-dataset/coco2017/val20...   
5  /kaggle/input/coco-2017-dataset/coco2017/val20...   
6  /kaggle/input/coco-2017-dataset/coco2017/val20...   
7  /kaggle/input/coco-2017-dataset/coco2017/val20...   
8  /kaggle/input/coco-2017-dataset/coco2017/val20...   
9  /kaggle/input/coco-2017-dataset/coco2017/val20...   

                                                text  
0  A busy intersection with cars waiting for a re...  
1  A couple of beds sitting next to each other in...  
2            a man at the beach trying to fly a kite  
3             A tree stands net to an old wood house  
4  A white plate topped with a pizza cut into 6 s...  
5       Graffiti on a building next to street signs. 

In [33]:
test_csv_path = f"dataset_test_{int(k*100)}.csv"
model_path = "clip_ft_final.pth"
metrics = test(test_csv_path, model_path, top_k=10)

Encoding Unique Images: 100%|██████████| 60/60 [00:32<00:00,  1.85it/s]



Unique Images Features Shape: torch.Size([3789, 512])


Encoding Texts: 100%|██████████| 98/98 [00:24<00:00,  3.99it/s]



Texts Features Shape: torch.Size([6253, 512])


Processing Queries: 100%|██████████| 6253/6253 [00:04<00:00, 1283.83it/s]



Metrics Saved To: clip_test_metrics_20251103_181708.json


In [34]:
print(f"\nTest Results on {k*100:.2f}% Test Dataset:")
for metric in metrics:
    if metric != "Results":
        print(f"\t{metric:15}: {metrics[metric]*100:.4f} %" )


Test Results on 25.00% Test Dataset:
	MRR            : 42.6440 %
	Precision@1    : 30.7532 %
	Recall@1       : 30.7532 %
	Precision@2    : 41.6920 %
	Recall@2       : 41.6920 %
	Precision@3    : 47.4172 %
	Recall@3       : 47.4172 %
	Precision@4    : 51.6072 %
	Recall@4       : 51.6072 %
	Precision@5    : 55.4934 %
	Recall@5       : 55.4934 %
	Precision@6    : 58.3240 %
	Recall@6       : 58.3240 %
	Precision@7    : 60.7069 %
	Recall@7       : 60.7069 %
	Precision@8    : 62.4500 %
	Recall@8       : 62.4500 %
	Precision@9    : 64.4011 %
	Recall@9       : 64.4011 %
	Precision@10   : 66.2882 %
	Recall@10      : 66.2882 %


In [35]:
import matplotlib.pyplot as plt

print("\nPrecision@K Plot:\n")
precision_keys = [k for k in metrics if k.lower().startswith("precision@")]
precision_vals = [metrics[k]*100 for k in precision_keys]
ks = [int(k.split("@")[-1]) for k in precision_keys]

plt.figure(figsize=(12,12))  
plt.plot(ks, precision_vals, marker='o', color='blue', linewidth=2)
plt.xlabel("k", fontsize=12)
plt.ylabel("Precision (%)", fontsize=12)
plt.title("Precision@k", fontsize=14)
plt.ylim(0, 100)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


Precision@K Plot:



<Figure size 1200x1200 with 1 Axes>

In [36]:
import matplotlib.pyplot as plt

print("\nRecall@K Plot:\n")
recall_keys = [k for k in metrics if k.lower().startswith("recall@")]
recall_vals = [metrics[k]*100 for k in recall_keys]
ks = [int(k.split("@")[-1]) for k in recall_keys]

plt.figure(figsize=(12,12))
plt.plot(ks, recall_vals, marker='o', color='blue', linewidth=2)
plt.xlabel("k")
plt.ylabel("Recall (%)")
plt.title("Recall@k")
plt.ylim(0, 100)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


Recall@K Plot:



<Figure size 1200x1200 with 1 Axes>

In [37]:
import matplotlib.pyplot as plt
from PIL import Image
import random

def display_query_results(metrics, num_queries=10):
    results = metrics["Results"]
    indices = list(range(len(results)))
    random.shuffle(indices)
    
    for idx in indices[:num_queries]:
        query_data = results[idx]
        query_text = query_data["query"]
        gt_path = query_data["gt_path"]
        gt_rank = query_data.get("gt_rank", None)
        gt_score = query_data.get("gt_score", None)
        retrieved = query_data["retrieved"]
        
        total_cols = 1 + len(retrieved)
        plt.figure(figsize=(3*total_cols, 5))
        plt.suptitle(f"Query: {query_text}", fontsize=12)
        
        plt.subplot(1, total_cols, 1)
        plt.imshow(Image.open(gt_path).convert("RGB"))
        plt.title(f"Ground Truth\nRank: {gt_rank}\nScore: {gt_score:.2f}")
        plt.axis('off')
        
        for j, r in enumerate(retrieved):
            plt.subplot(1, total_cols, j+2)
            plt.imshow(Image.open(r["path"]).convert("RGB"))
            plt.title(f"Rank: {r['rank']}\nScore: {r['score']:.2f}")
            plt.axis('off')
        
        plt.show()

In [38]:
display_query_results(metrics, 25)

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

... (15 outputs left out)

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

<Figure size 3300x500 with 11 Axes>

In [39]:
# import os

# dir_path = "/kaggle/working/"

# for root, _, files in os.walk(dir_path):
#     for f in files:
#         file_path = os.path.join(root, f)
#         answer = input(f"\nRemove {file_path}? (y/n): ").strip().lower()
#         if answer == '1':
#             os.remove(file_path)
#             print(f"\tRemoved: {file_path}")
#         else:
#             print(f"\tKept: {file_path}")